In [1]:
from semanticscholar import SemanticScholar
from crossref.restful import Works
from itertools import product
import pandas as pd
import json
import dblp
import requests

In [2]:
crossref_work = Works()
sch = SemanticScholar()

In [3]:
# search params
primary_keywords = [
    "Generative AI",
    "LLM",
    "LM",
    "GenAI",
    "large language model",
    "language model",
    "codex",
    "gpt",
    "gpt-3",
    "gpt-4",
]
secondary_keywords = [
    "overreliance",
    "misinformation",
    "accessbility",
    "privacy",
    "enviromental",
    "explainability",
    "trustworthy",
    "responsible",
]
others = ["mitigation", "ethics", "societal", "social", "ethical"]


year = "2019-"
years = ["2019", "2020", "2021", "2022", "2023"]
dblp_formatted_years = " " + "|".join(f"{year}" for year in years)

all_combinations = list(product(primary_keywords, secondary_keywords, others))

In [6]:
# Prepare DataFrame to store the results
columns = [
    'PaperTitle',
    'DOI',    
    'SearchString',
    'SearchedFrom',
]
search_results_df = pd.DataFrame(columns=columns)

In [7]:
# SemanticScholar fields
fields = [
    "title",
    "externalIds",
    "paperId",
]

In [9]:
try:
    for search_string in all_combinations:
        search_string_formatted = ' + '.join(search_string)
        print(f"-------------------------------searching for {search_string_formatted}------------------------------")
        
        # Semantic Scholar
         # send http request to api
        api_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
        headers = {"Content-Type": "application/json"}
        params = {"query": search_string_formatted, "year": year, "fields": ",".join(fields)}
        response = requests.get(api_url, headers=headers, params=params)

        # parse response
        response_json = response.json()
        sch_results = response_json["data"]
        print(f"Semantic scholar total: {response_json['total']}")
        # Add results to DataFrame
        sch_count = 0
        for result in sch_results:
            sch_count += 1
            new_paper = {
                'PaperTitle': result['title'],
                'DOI': result['externalIds'].get('DOI'),
                'SearchString': search_string_formatted,
                'SearchedFrom': 'Semantic Scholar'
            }
            print(f"{sch_count}. sch process paper: ", new_paper)
            search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 
        
        # # Crossref
        # # TODO: too many result
        # cr_search_results = crossref_work.query(search_string_formatted).filter(from_online_pub_date=year)
        # print(f"Crossref total: {cr_search_results.count()}")
        # crossref_count = 0
        # for cr_result in cr_search_results:
        #     crossref_count += 1
        #     new_paper = {
        #         'PaperTitle': cr_result.get('title'),
        #         'DOI': cr_result.get('DOI'),
        #         'SearchString': search_string_formatted,
        #         'SearchedFrom': 'Crossref'
        #     }
        #     print(f"{crossref_count}. Crossref process paper: ", new_paper)
        #     search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True) 

        # # search in dblp
        # dblp_search_results = dblp.search(search_string_formatted + dblp_formatted_years)
        # print(f"DBLP total: {len(dblp_search_results)}")
        # dblp_count = 0
        # for key, result in dblp_search_results.items():
        #     dblp_count += 1
        #     new_paper = {
        #         'PaperTitle': result.get('title'),
        #         'DOI': result.get('doi'),
        #         'SearchString': search_string_formatted,
        #         'SearchedFrom': 'DBLP'
        #     }
        #     print(f"{dblp_count}. DBLP process paper: ", new_paper)
        #     search_results_df = pd.concat([search_results_df, pd.DataFrame([new_paper])], ignore_index=True)
        # print(f"^^^^^^^^^^^^^^^^^^^^^^^^^^^searching end for {search_string_formatted}^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
    search_results_df.to_csv('data/search_results.csv')
except Exception as e:
    print(f"An error occurred: {e.with_traceback()}")
    search_results_df.to_csv('data/search_results.csv')

    
    
    
    

-------------------------------searching for Generative AI + overreliance + mitigation------------------------------
Semantic scholar total: 0
Crossref total: 82106
1. Crossref process paper:  {'PaperTitle': ['Generative AI with Personalities'], 'DOI': '10.1007/978-1-4842-9579-3_3', 'SearchString': 'Generative AI + overreliance + mitigation', 'SearchedFrom': 'Crossref'}
2. Crossref process paper:  {'PaperTitle': ['Prototyping with Generative AI'], 'DOI': '10.1007/978-1-4842-9579-3_5', 'SearchString': 'Generative AI + overreliance + mitigation', 'SearchedFrom': 'Crossref'}
3. Crossref process paper:  {'PaperTitle': ['Dilemmas Interacting with Generative AI'], 'DOI': '10.1007/978-1-4842-9579-3_11', 'SearchString': 'Generative AI + overreliance + mitigation', 'SearchedFrom': 'Crossref'}
4. Crossref process paper:  {'PaperTitle': ['Generative AI Form and Composition'], 'DOI': '10.1007/978-1-4842-9579-3_7', 'SearchString': 'Generative AI + overreliance + mitigation', 'SearchedFrom': 'Crossr

KeyboardInterrupt: 

# Testing stuff

In [38]:
search_string = 'Generative AI AND Social Impact AND education'
test =  cr_search_results = crossref_work.query(search_string).count()

In [39]:
search_string = 'generative ai' + " 2020|2021"
dblp_test = dblp.search(search_string)
dblp_test.items()

dict_items([(0, {'title': 'Joint Proceedings of the Workshops on Human-AI Co-Creation with Generative Models and User-Aware Conversational Agents co-located with 25th International Conference on Intelligent User Interfaces (IUI 2020), Cagliari, Italy, March 17, 2020.', 'year': '2021', 'venue': ['HAI-GEN+user2agent@IUI', 'CEUR Workshop Proceedings'], 'doi': None, 'url': 'https://ceur-ws.org/Vol-2848', 'bibtex': 'https://dblp.org/rec/conf/iui/2020hai?view=bibtex'}), (1, {'title': 'Generative adversarial network-based rogue device identification using differential constellation trace figure.', 'year': '2021', 'venue': 'EURASIP J. Wirel. Commun. Netw.', 'doi': '10.1186/S13638-021-01950-2', 'url': 'https://doi.org/10.1186/s13638-021-01950-2', 'bibtex': 'https://dblp.org/rec/journals/ejwcn/ChenPHF21?view=bibtex'}), (2, {'title': 'View synthesis-based light field image compression using a generative adversarial network.', 'year': '2021', 'venue': 'Inf. Sci.', 'doi': '10.1016/J.INS.2020.07.073